<a href="https://colab.research.google.com/github/SyameimaruKoa/Minecraft-mod-AutoTranslation-tool/blob/main/run_colab_server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Minecraft Mod Auto Translator — Google Colab（LM Link）

ColabでLM Studio（llmster）を動かし、LM Link経由でPCと接続するのじゃ。認証情報は`dbv4-wd14-tagger`と共通のGoogleドライブフォルダーへ保存する。モデルはpfnet公式またはmradermacherの8B通常量子化・i1量子化を選べる。既定はpfnet公式8B／Q4_K_Mじゃ。

## 1. 共通設定

Googleドライブの保存先、LM Studioの再ログイン、PLaMoのモデルサイズ・量子化形式・同時推論数を先にまとめて選択するのじゃ。

In [ ]:
#@title ⚙️ すべての設定
#@markdown **Googleドライブ**（dbv4-wd14-taggerと共通の保存先）

USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_FOLDER_NAME = ".colab-secret" #@param {type:"string"}

#@markdown **LM Studioの認証**
FORCE_RELOGIN = False #@param {type:"boolean"}

#@markdown **PLaMo 3 Translateのモデル**（同時ロードは1つのみ）
model_source = "pfnet" # @param ["pfnet", "mradermacher", "mradermacher-i1"]
plamo_size = "8b" # @param ["0.8b", "2b", "8b", "31b"]
quantization = "Q4_K_M" # @param ["Q4_K_M", "Q3_K_M", "Q2_K", "Q4_0", "Q4_K_S", "Q5_K_S", "Q5_K_M", "Q6_K", "Q8_0", "IQ4_XS", "IQ4_NL", "IQ3_M", "IQ3_S", "F16"]
PARALLEL_REQUESTS = 4 # @param {type:"integer"}
#@markdown **LM Linkクラスター識別名**（空欄なら標準ID。ほかのPCと異なる名前にすると個別に並列利用できる）
MODEL_IDENTIFIER = "" # @param {type:"string"}

if not 1 <= PARALLEL_REQUESTS <= 16:
    raise ValueError("同時推論数は1から16で指定するのじゃ")

CONFIG_DRIVE_FOLDER = DRIVE_FOLDER_NAME.strip("/ ") if DRIVE_FOLDER_NAME else ".colab-secret"
if (
    CONFIG_DRIVE_FOLDER in (".", "..")
    or "/" in CONFIG_DRIVE_FOLDER
    or "\\" in CONFIG_DRIVE_FOLDER
):
    raise ValueError("Googleドライブの保存先は単一フォルダー名を指定するのじゃ")
print(
    "[INFO] Googleドライブ保存: "
    + (f"有効 (MyDrive/{CONFIG_DRIVE_FOLDER})" if USE_GOOGLE_DRIVE else "無効")
)

## 2. GPU別のPLaMo実行目安

**Colabが割り当てるGPUは毎回変わる**ため、次のセルでGPU名と現在のVRAM総量・使用量・空き容量を実測する。

| GPU（公称VRAM） | 推奨するGGUF | 31Bの目安 |
|---|---|---|
| T4（16GB） | 公式8B／Q4_K_M、余裕があればmradermacher通常版8B／Q8_0 | 全レイヤーGPUロードは推奨しない |
| L4（24GB） | 0.8B / 2B / 8B：Q4_K_M | Q2_K / Q3_K_Mは候補。Q4_K_Mは余裕が小さく要検証 |
| A100（40GB） | 全サイズ：Q4_K_M | Q4_K_Mを候補にできる |
| A100（80GB）・H100（80GB） | 全サイズ：Q4_K_M | Q4_K_Mを候補にできる |

**PLaMo公式GGUFのファイル容量（十進GB）**

| サイズ | Q4_K_M | Q4_0 | Q3_K_M | Q2_K |
|---|---:|---:|---:|---:|
| 0.8B | 0.57 | 0.55 | 0.49 | 0.41 |
| 2B | 1.81 | 1.70 | 1.50 | 1.23 |
| 8B | 5.28 | ― | 4.31 | 3.45 |
| 31B | 19.53 | ― | 15.86 | 12.61 |

**追加の8B量子化：** [mradermacher 通常版](https://huggingface.co/mradermacher/plamo-3-translate-8b-GGUF)はQ5_K_M（約6.1GB）・Q6_K（約7.0GB）・Q8_0（約9.1GB）などに対応。[mradermacher i1版](https://huggingface.co/mradermacher/plamo-3-translate-8b-i1-GGUF)はimatrix量子化を含む。mradermacherは8Bのみ選択可能とし、配布ファイル一覧に存在する形式だけを取得する。通常版Q8_0はT4（16GB）の精度優先候補、i1-Q5_K_MはRTX 2070（8GB）の候補じゃ。速度・実際の品質差は未測定じゃ。

**判定方法：** 下のセルは空きVRAMとファイル容量を比較し、4GiB以上の余裕があれば「容量上の候補」、本体だけ収まれば「余裕小」、本体も収まらなければ「GPU全量は困難」と表示する。この4GiBは便宜的な目安で、**実測の必要VRAMではない**。実際のGPU使用量はLM Studioのコンテキスト長・並列数・KVキャッシュ・GPUオフロードによって変わる。CPUオフロードで動く場合もある。**候補表示はロード成功や4並列動作の保証ではない**。

公式資料：[0.8B](https://huggingface.co/pfnet/plamo-3-translate-0.8b-GGUF) / [2B](https://huggingface.co/pfnet/plamo-3-translate-2b-GGUF) / [8B](https://huggingface.co/pfnet/plamo-3-translate-8b-GGUF) / [31B](https://huggingface.co/pfnet/plamo-3-translate-31b-GGUF)、[NVIDIA GPU仕様](https://www.nvidia.com/ja-jp/data-center/graphics-cards-for-virtualization/)、[ColabのGPU割当について](https://research.google.com/colaboratory/faq.html?hl=ja)。


In [ ]:
#@title GPU・VRAMの検出とモデル容量の目安
import csv
import io
import subprocess

PLAMO_GGUF_SIZES_GB = {
    "0.8b": {"Q4_K_M": 0.57, "Q4_0": 0.55, "Q3_K_M": 0.49, "Q2_K": 0.41},
    "2b": {"Q4_K_M": 1.81, "Q4_0": 1.70, "Q3_K_M": 1.50, "Q2_K": 1.23},
    "8b": {"Q4_K_M": 5.28, "Q3_K_M": 4.31, "Q2_K": 3.45},
    "31b": {"Q4_K_M": 19.53, "Q3_K_M": 15.86, "Q2_K": 12.61},
}
MRADERMACHER_8B_GGUF_SIZES_GB = {
    "Q4_K_M": 5.3, "Q5_K_S": 5.9, "Q5_K_M": 6.1,
    "Q6_K": 7.0, "Q8_0": 9.1, "IQ4_XS": 4.8,
}
if model_source != "pfnet":
    PLAMO_GGUF_SIZES_GB = {"8b": MRADERMACHER_8B_GGUF_SIZES_GB}

GPU_VRAM_MARGIN_GIB = 4.0

try:
    gpu_output = subprocess.run(
        ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.used,memory.free",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=True, timeout=15, encoding="utf-8"
    ).stdout
except (FileNotFoundError, subprocess.CalledProcessError, subprocess.TimeoutExpired) as error:
    print(f"[WARN] NVIDIA GPU情報を取得できぬ: {error}", flush=True)
    gpu_output = ""

gpu_rows = list(csv.reader(io.StringIO(gpu_output), skipinitialspace=True))
if not gpu_rows:
    print("[WARN] NVIDIA GPUを検出できぬ。ランタイムのGPU設定を確認するのじゃ。", flush=True)
for row in gpu_rows:
    if len(row) != 5:
        print(f"[WARN] nvidia-smiの出力形式が想定外じゃ: {row}", flush=True)
        continue
    try:
        gpu_index, gpu_name = row[0].strip(), row[1].strip()
        total_mib, used_mib, free_mib = [int(v.strip()) for v in row[2:]]
    except ValueError as error:
        print(f"[WARN] GPUのVRAMを解析できぬ: {error}", flush=True)
        continue
    free_gib = free_mib / 1024
    print(
        f"GPU {gpu_index}: {gpu_name} | 総VRAM {total_mib / 1024:.2f} GiB | "
        f"使用 {used_mib / 1024:.2f} GiB | 空き {free_gib:.2f} GiB",
        flush=True
    )
    print(" PLaMo / 量子化       GGUF容量    GPU全量ロードの容量目安", flush=True)
    for size, variants in PLAMO_GGUF_SIZES_GB.items():
        for quant, gb in variants.items():
            weight_gib = gb * (10 ** 9) / (1024 ** 3)
            if free_gib >= weight_gib + GPU_VRAM_MARGIN_GIB:
                status = "容量上の候補（実行保証なし）"
            elif free_gib >= weight_gib:
                status = "余裕小・要検証"
            else:
                status = "GPU全量は困難（CPUオフロードは別）"
            chosen = " ← 選択中" if size == plamo_size and quant == quantization else ""
            print(f" {size:>4} / {quant:<7}      {gb:>5.2f} GB    {status}{chosen}", flush=True)
print("[INFO] 非公式GGUF容量は概算じゃ。4GiBの余裕は便宜的な比較基準で、実VRAM必要量はロード時に確認するのじゃ。", flush=True)


## 3. LM Studioのインストール

先にインストールし、次のセルで認証情報を復元してからデーモンを起動する。

In [ ]:
#@title LM Studioのインストール
import codecs
import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

def run_console(command, check=True):
    if shutil.which("script") is None:
        raise RuntimeError("PTY用のscriptコマンドが見つからぬ。util-linuxを確認するのじゃ")
    argv = [str(item) for item in command]
    print("\n$ " + shlex.join(argv), flush=True)
    process = subprocess.Popen(
        ["script", "-q", "-f", "-e", "-c", shlex.join(argv), "/dev/null"],
        stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        bufsize=0
    )
    decoder = codecs.getincrementaldecoder("utf-8")("replace")
    chunks = []
    try:
        while True:
            data = process.stdout.read(8192)
            if not data:
                break
            decoded = decoder.decode(data)
            chunks.append(decoded)
            sys.stdout.write(decoded)
            sys.stdout.flush()
        decoded = decoder.decode(b"", final=True)
        chunks.append(decoded)
        sys.stdout.write(decoded)
        sys.stdout.flush()
        code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=3)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        process.stdout.close()
        process.stdin.close()
    print(f"[EXIT] {code}: {shlex.join(argv)}", flush=True)
    if check and code:
        raise subprocess.CalledProcessError(code, argv)
    return subprocess.CompletedProcess(argv, code, stdout="".join(chunks))

run_console(["pkill", "-x", "llmster"], check=False)
run_console(["pkill", "-x", "lmstudio"], check=False)
run_console(["bash", "-o", "pipefail", "-c", "curl -fL https://lmstudio.ai/install.sh | bash"])
os.environ["PATH"] = str(Path.home() / ".lmstudio" / "bin") + os.pathsep + os.environ["PATH"]
run_console(["lms", "bootstrap"])
run_console(["sync"])
print("LM Studioのセットアップが完了したのじゃ。")


## 4. Googleドライブへのログイン状態保存

DBV4のTailscaleと同様、LM Studioのログイン状態をGoogleドライブの `MyDrive/.colab-secret/lmstudio/` 以下へ保存・自動復元する。パスフレーズの入力やColabシークレットは不要じゃ。保存対象は `credentials` と `hub` の認証状態のみで、内部デーモンキーやモデル本体は保存しない。

In [ ]:
#@title 認証情報の復元と保存
import os
import shutil
from pathlib import Path

from google.colab import drive

LM_HOME = Path.home() / ".lmstudio"
DRIVE_AUTH_DIR = None
AUTH_DIRS = ("credentials", "hub")

if USE_GOOGLE_DRIVE:
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    DRIVE_AUTH_DIR = Path("/content/drive/MyDrive") / CONFIG_DRIVE_FOLDER / "lmstudio"

def copy_login_files(source, destination):
    copied = 0
    for directory in AUTH_DIRS:
        root = source / directory
        if not root.is_dir() or root.is_symlink():
            continue
        for original in root.rglob("*"):
            if not original.is_file() or original.is_symlink():
                continue
            relative = original.relative_to(source)
            target = destination / relative
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(original, target)
            os.chmod(target, 0o600)
            copied += 1
    return copied

def restore_login():
    if not USE_GOOGLE_DRIVE:
        return False
    restored = copy_login_files(DRIVE_AUTH_DIR, LM_HOME)
    if restored:
        print(f"[INFO] GoogleドライブからLM Link認証情報を復元: {restored}ファイル")
    else:
        print("[INFO] 保存済みのLM Link認証情報なし。初回ログインが必要じゃ。")
    return bool(restored)

def backup_login():
    if not USE_GOOGLE_DRIVE:
        return
    saved = copy_login_files(LM_HOME, DRIVE_AUTH_DIR)
    if not saved:
        print("[WARN] LM Linkは有効じゃが、認証ファイルが見つからずバックアップできなかったのじゃ。")
        print("[WARN] Colabのランタイムを作り直した際、再ログインが必要になる可能性があるぞ。")
        for directory_name in ("credentials", "hub", ".internal"):
            root = LM_HOME / directory_name
            count = sum(1 for entry in root.rglob("*") if entry.is_file() and not entry.is_symlink()) if root.is_dir() else 0
            print(f"[INFO] 保存候補 {directory_name}: {count}個の通常ファイル")
        for directory in (LM_HOME, Path.home() / ".config" / "LM Studio", Path.home() / ".cache" / "lm-studio"):
            if directory.is_dir():
                names = sorted(item.name for item in directory.iterdir() if item.is_dir() and not item.is_symlink())
                print(f"[INFO] 認証保存先の調査: {directory} / サブディレクトリー: {names}")
        return False
    print(f"[INFO] LM Link認証情報をGoogleドライブに保存: {saved}ファイル")
    return True

auth_restored = restore_login()

## 5. LM Linkのログインと接続

保存済み認証情報が有効なら再ログインせずに接続する。初回またはログイン期限切れなら `lms login` の認証URLをブラウザで開くのじゃ。ログイン後に認証情報を保存する。

最初の「すべての設定」で `FORCE_RELOGIN` をオンにすると、一度ログアウトしてブラウザ認証をやり直せる。認証ファイルの保存先は調査中じゃ。

In [ ]:
#@title LM Linkログインと接続
import subprocess
import time

run_console(["lms", "daemon", "up"])
time.sleep(3)
whoami = run_console(["lms", "whoami"], check=False)
if FORCE_RELOGIN and whoami.returncode == 0:
    print("LM Studioから一度ログアウトし、再認証するのじゃ。", flush=True)
    run_console(["lms", "logout"])
if whoami.returncode != 0 or FORCE_RELOGIN:
    print("LM Studioへのログインが必要じゃ。表示される認証URLで承認するのじゃ。", flush=True)
    run_console(["lms", "login"])
else:
    print("現在のLM Studioログインは有効じゃ。再認証にはFORCE_RELOGINを有効にするのじゃ。")
run_console(["lms", "link", "enable"])
backup_login()
run_console(["lms", "link", "status"])
print("LM Linkの接続処理が完了したのじゃ。")


## 6. PLaMo 3 Translateモデルのダウンロードとロード

最初の「すべての設定」でモデルを選択する。既定は**pfnet公式8B／Q4_K_M**。公式は0.8B・2B・8B・31Bに対応。追加のmradermacher通常版・i1版は8Bのみ対応し、Hugging Faceのファイル一覧に存在する形式だけ指定する。`lms get`の検索処理を使わず、選択した配布元のGGUFファイルを直接取得するため、問題のコマンドは実行しない。ダウンロード中は進捗バー・転送量を表示し、中断時は可能なら途中から再開する。モデルの読込はLM Linkを一時停止してColab内のモデルだけを一覧化し、GGUFファイルのパスに対応する実際の`modelKey`を取得して実行する。読込後にLM Linkを再有効化する。Colabランタイムを破棄するとモデルファイルは消えるが、Googleドライブの保存済み認証情報は残る。31BはGPUメモリ不足でロードできない場合があるのじゃ。

Colabでロードする際の同時推論数は最初の設定欄で指定し、ロード後にローカルAPIが返す`config.parallel`もログに表示する。

In [ ]:
#@title PLaMoモデルの選択と読み込み
import os
import re
import subprocess
from pathlib import Path
from urllib.parse import quote

OFFICIAL_GGUF_FORMATS = {
    "0.8b": ("Q4_K_M", "Q4_0", "Q3_K_M", "Q2_K"),
    "2b": ("Q4_K_M", "Q4_0", "Q3_K_M", "Q2_K"),
    "8b": ("Q4_K_M", "Q3_K_M", "Q2_K"),
    "31b": ("Q4_K_M", "Q3_K_M", "Q2_K"),
}
if model_source not in ("pfnet", "mradermacher", "mradermacher-i1"):
    raise ValueError("未対応のGGUF配布元じゃ")
if plamo_size not in OFFICIAL_GGUF_FORMATS:
    raise ValueError("未対応のPLaMoサイズじゃ")
if model_source != "pfnet" and plamo_size != "8b":
    raise ValueError("mradermacherの追加配布は8Bのみ対応じゃ")

repo_name = f"plamo-3-translate-{plamo_size}-GGUF"
filename = f"plamo-3-translate-{plamo_size}-{quantization}.gguf"
source_owner = "pfnet" if model_source == "pfnet" else "mradermacher"
if model_source == "mradermacher-i1":
    repo_name = "plamo-3-translate-8b-i1-GGUF"
source_repo = f"{source_owner}/{repo_name}"

def resolve_hf_gguf_file(file_entries, size, quant, i1):
    prefix = f"plamo-3-translate-{size}"
    suffix = re.compile(r"[._-]" + re.escape(quant) + r"\.gguf$", re.IGNORECASE)
    candidates = []
    for entry in file_entries:
        if not isinstance(entry, dict):
            continue
        path = entry.get("path") or entry.get("rfilename")
        if not isinstance(path, str) or not path.lower().endswith(".gguf"):
            continue
        base = path.rsplit("/", 1)[-1].lower()
        if not base.startswith(prefix) or not suffix.search(base):
            continue
        if i1 != (".i1-" in base or "-i1-" in base):
            continue
        candidates.append((path, entry.get("size")))
    if len(candidates) != 1:
        available = sorted(
            (entry.get("path") or entry.get("rfilename") or "")
            for entry in file_entries
            if isinstance(entry, dict) and str(entry.get("path") or entry.get("rfilename") or "").lower().endswith(".gguf")
        )
        raise ValueError(
            f"選択した量子化GGUFを一意に取得できぬ: {size}/{quant} "
            f"(候補={len(candidates)})。配布中: {available}"
        )
    return candidates[0]

if model_source == "pfnet":
    if quantization not in OFFICIAL_GGUF_FORMATS[plamo_size]:
        raise ValueError(f"pfnet公式で配布されていない組み合わせじゃ: {plamo_size}/{quantization}")
else:
    import requests
    response = requests.get(
        f"https://huggingface.co/api/models/{source_repo}/tree/main",
        params={"recursive": "true", "limit": 1000}, timeout=(10, 30)
    )
    response.raise_for_status()
    entries = response.json()
    if not isinstance(entries, list):
        raise ValueError("Hugging Faceのファイル一覧が取得できぬ")
    filename, expected_bytes = resolve_hf_gguf_file(
        entries, plamo_size, quantization, model_source == "mradermacher-i1"
    )
    if not isinstance(expected_bytes, int) or expected_bytes <= 100_000_000:
        raise ValueError("Hugging FaceのGGUF容量が確認できぬ")

model_dir = Path.home() / ".lmstudio" / "models" / source_owner / repo_name
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / filename
model_path.parent.mkdir(parents=True, exist_ok=True)
partial_path = model_dir / (filename + ".part")
model_id = MODEL_IDENTIFIER.strip() or f"plamo-3-translate-{plamo_size}"
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]*", model_id):
    raise ValueError("モデルIDは英数字で始まり、英数字・ハイフン・アンダースコア・ピリオドのみ使用できます")
if model_source == "pfnet":
    url = f"https://huggingface.co/pfnet/{repo_name}/resolve/main/{filename}?download=true"
else:
    url = f"https://huggingface.co/{source_repo}/resolve/main/{quote(filename)}?download=true"
expected_path = f"{source_repo}/{filename}"

if (
    not model_path.exists()
    or model_path.stat().st_size < 100_000_000
    or (model_source != "pfnet" and model_path.stat().st_size != expected_bytes)
):
    print(f"GGUFを取得: {expected_path}", flush=True)
    import re
    import time
    import requests
    from tqdm.auto import tqdm

    def download_with_progress(source_url, partial, complete):
        for attempt in range(1, 5):
            offset = partial.stat().st_size if partial.exists() else 0
            headers = {"Range": f"bytes={offset}-"} if offset else {}
            try:
                with requests.get(source_url, headers=headers, stream=True, timeout=(30, 120)) as response:
                    if response.status_code == 416 and offset:
                        match = re.fullmatch(r"bytes \*/(\d+)", response.headers.get("Content-Range", ""))
                        if match and int(match.group(1)) == offset:
                            os.replace(partial, complete)
                            return
                    response.raise_for_status()
                    if response.status_code == 206:
                        match = re.fullmatch(
                            r"bytes (\d+)-(\d+)/(\d+|\*)",
                            response.headers.get("Content-Range", "")
                        )
                        if not match or int(match.group(1)) != offset:
                            raise RuntimeError("ダウンロードの再開位置が一致しないのじゃ")
                        total = int(match.group(3)) if match.group(3) != "*" else None
                        mode = "ab"
                    elif response.status_code == 200:
                        if offset:
                            print("サーバーが再開を受け付けなかったため、先頭から再取得するのじゃ。", flush=True)
                        offset = 0
                        total = int(response.headers["Content-Length"]) if response.headers.get("Content-Length") else None
                        mode = "wb"
                    else:
                        raise RuntimeError(f"予期しないダウンロード応答: HTTP {response.status_code}")
                    print(f"ダウンロード開始: {offset / 1048576:.1f} MiB取得済み", flush=True)
                    with partial.open(mode) as output, tqdm(
                        total=total, initial=offset, unit="B", unit_scale=True,
                        desc="PLaMo GGUF", mininterval=0.5, leave=True
                    ) as progress:
                        for chunk in response.iter_content(chunk_size=1048576):
                            if chunk:
                                output.write(chunk)
                                progress.update(len(chunk))
                    current = partial.stat().st_size
                    if total is not None and current != total:
                        raise RuntimeError(f"サイズ不一致: {current} / {total}バイト")
                    if current < 100_000_000:
                        raise RuntimeError("GGUFのダウンロード結果が小さすぎるのじゃ")
                    if model_source != "pfnet" and current != expected_bytes:
                        raise RuntimeError(f"配布元のGGUF容量と一致せぬ: {current} / {expected_bytes}バイト")
                    os.replace(partial, complete)
                    print("GGUFのダウンロードが完了したのじゃ。", flush=True)
                    return
            except (requests.RequestException, OSError) as error:
                if attempt == 4:
                    raise
                print(f"通信が中断したのじゃ（{attempt}/4）: {error}。途中から再試行するぞ。", flush=True)
                time.sleep(min(attempt * 2, 8))
        raise RuntimeError("GGUFのダウンロードに失敗したのじゃ")

    download_with_progress(url, partial_path, model_path)
else:
    print(f"ダウンロード済みGGUFを再利用: {filename}", flush=True)

import requests
import time

LMSTUDIO_LOCAL_API = "http://127.0.0.1:1234/api/v1/models"

def read_local_models():
    for attempt in range(1, 4):
        try:
            response = requests.get(LMSTUDIO_LOCAL_API, timeout=(5, 20))
            response.raise_for_status()
            models = response.json().get("models")
            if not isinstance(models, list):
                raise ValueError("models配列が返されていないのじゃ")
            return models
        except (requests.RequestException, ValueError, AttributeError) as error:
            if attempt == 3:
                raise RuntimeError(
                    f"ColabのLM Studioモデル一覧APIが利用できぬ: {error}"
                ) from error
            print(f"[WARN] モデル一覧APIの取得失敗（{attempt}/3）: {error}", flush=True)
            time.sleep(attempt * 2)

def resolve_local_model_key(models, size, quant, file_bytes):
    family = f"plamo-3-translate-{size}"
    candidates = []
    for item in models:
        if not isinstance(item, dict):
            continue
        key = item.get("key")
        if item.get("type") != "llm" or not isinstance(key, str):
            continue
        if family not in key.casefold() or item.get("format", "gguf") != "gguf":
            continue
        variant = item.get("quantization")
        if (
            not isinstance(variant, dict)
            or str(variant.get("name", "")).casefold().removeprefix("i1-") != quant.casefold()
        ):
            continue
        if item.get("size_bytes") != file_bytes:
            continue
        selected = item.get("selected_variant")
        variants = item.get("variants")
        if selected is not None and (
            not isinstance(selected, str)
            or (isinstance(variants, list) and selected not in variants)
        ):
            continue
        candidates.append(selected or key)
    if len(candidates) != 1:
        known = [
            {
                "key": item.get("key"),
                "size_bytes": item.get("size_bytes"),
                "quantization": item.get("quantization")
            }
            for item in models
            if isinstance(item, dict)
            and family in str(item.get("key", "")).casefold()
        ]
        raise RuntimeError(
            f"ColabのローカルGGUFを一意に特定できぬ: {family}/{quant} "
            f"({file_bytes} bytes)。API候補: {known}"
        )
    return candidates[0]

def unload_local_models_with_retry(max_attempts=6):
    retryable_errors = ("connector_closed", "connection closed", "already been unloaded")
    for attempt in range(1, max_attempts + 1):
        result = run_console(["lms", "unload", "--all"], check=False)
        if result.returncode == 0:
            return
        message = (result.stdout or "").casefold()
        if not any(marker in message for marker in retryable_errors):
            raise subprocess.CalledProcessError(
                result.returncode, result.args, output=result.stdout
            )
        if attempt == max_attempts:
            raise RuntimeError(
                f"LM Studioのアンロードに{max_attempts}回連続で失敗したのじゃ。"
                "デーモンの接続状態を確認してくれ。"
            )
        delay = min(attempt * 2, 10)
        print(
            f"[WARN] LM Link切断処理との競合（{attempt}/{max_attempts}）。"
            f"{delay}秒後にアンロードを再試行するのじゃ。",
            flush=True
        )
        time.sleep(delay)

print("[INFO] LM Linkを一時停止し、Colabのローカルモデルだけを読み込むのじゃ。", flush=True)
run_console(["lms", "link", "disable"])
try:
    unload_local_models_with_retry()
    run_console(["lms", "server", "start", "--port", "1234"])
    inventory = read_local_models()
    model_key = resolve_local_model_key(
        inventory, plamo_size, quantization, model_path.stat().st_size
    )
    print(f"[INFO] LM StudioのローカルAPIで特定したモデルキー: {model_key}", flush=True)
    print(f"[INFO] 対象GGUF: {expected_path}", flush=True)
    run_console([
        "lms", "load", model_key, "--gpu", "max",
        "--parallel", str(PARALLEL_REQUESTS), "--identifier", model_id, "-y"
    ])
    loaded = read_local_models()
    instances = [
        (item, instance)
        for item in loaded
        if isinstance(item, dict)
        for instance in item.get("loaded_instances", [])
        if isinstance(instance, dict) and instance.get("id") == model_id
    ]
    if len(instances) != 1:
        raise RuntimeError(f"ロードしたモデルのインスタンスを確認できぬ: {model_id}")
    entry, instance = instances[0]
    actual_quant = entry.get("quantization") or {}
    if (
        entry.get("size_bytes") != model_path.stat().st_size
        or not isinstance(actual_quant, dict)
        or str(actual_quant.get("name", "")).casefold().removeprefix("i1-") != quantization.casefold()
    ):
        raise RuntimeError(
            f"ロードされたモデルが選択したGGUFと一致せぬ: {entry.get('key')}"
        )
    actual_config = instance.get("config") or {}
    actual_parallel = actual_config.get("parallel")
    print(
        f"[INFO] GPU KVキャッシュ: {actual_config.get('offload_kv_cache_to_gpu')}, "
        f"Flash Attention: {actual_config.get('flash_attention')}",
        flush=True
    )
    if actual_config.get("offload_kv_cache_to_gpu") is False:
        print("[WARN] KVキャッシュのGPU配置が無効じゃ。LM Studioのロード設定を確認してくれ。", flush=True)
    gpu_state = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=memory.total,memory.used,memory.free,utilization.gpu",
            "--format=csv,noheader,nounits"
        ],
        capture_output=True, text=True, check=False
    ) if shutil.which("nvidia-smi") else None
    if gpu_state is not None and gpu_state.returncode == 0:
        print(
            "[INFO] GPU VRAM/使用率 (合計MiB, 使用MiB, 空きMiB, GPU%): "
            + gpu_state.stdout.strip(), flush=True
        )
    print(
        f"[INFO] Colabの同時推論設定: 指定={PARALLEL_REQUESTS}, "
        f"API報告値={actual_parallel}", flush=True
    )
    if isinstance(actual_parallel, int) and actual_parallel != PARALLEL_REQUESTS:
        raise RuntimeError(
            f"並列数が反映されておらぬ: 指定={PARALLEL_REQUESTS}、実際={actual_parallel}"
        )
    if actual_parallel is None:
        print("[WARN] APIがparallel値を返さぬ。エンジンログでの検証が必要じゃ。", flush=True)
    print(f"[INFO] ローカルのGGUFをロードできたのじゃ: {model_id}", flush=True)
finally:
    print("[INFO] LM Linkを再び有効にするのじゃ。", flush=True)
    run_console(["lms", "link", "enable"])

print(f"ロード完了: {model_id}（LM Linkから利用可能）", flush=True)
